# Whale Instance Segmentation — YOLOv8 on Thermal Grayscale Images

This notebook documents the full pipeline for training and evaluating a YOLOv8 segmentation model on thermal drone images of whales.

**Pipeline overview:**
1. Dataset preparation — RGB → Grayscale conversion
2. Training (with best tuned model)
3. Final evaluation

---
## 0. Imports & Global Configuration

In [8]:
# ── Libraries ──────────────────────────────────────────────────────────
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
import shutil
import re
from PIL import Image
from ultralytics import YOLO
from pathlib import Path

In [2]:
zip_file = "./Flukeprint_detect_behav.v1-2026-05-10-behav.yolov8.zip"
new_name = "05_10_behav_dataset"

# Extract
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(new_name)

print(f"Extracted to {new_name}")

Extracted to 05_10_behav_dataset


In [11]:
# ── Dataset paths ─────────────────────────────────────────────────────────────
DATASET_ROOT  = Path("./05_10_behav_dataset")
DATASET_YAML      = DATASET_ROOT / "data.yaml"

# ── Model weights ─────────────────────────────────────────────────────────────
MODEL_TUNED_PATH    = "./hyper_tuned_best.pt"     # Final tuned model

# ── Model parameters ─────────────────────────────────────────────────────────────
MODEL_PARAMS_PATH = "./best_hyperparameters.yaml"

# ── Inference thresholds ─────────────────────────
CONF_THRESHOLD = 0.25  # Confidence threshold
IOU_THRESHOLD  = 0.30  # IoU threshold — intentionally permissive for partial whale detections

---
## 1. Dataset Preparation

### RGB → Grayscale 

The thermal drone images are inherently black-and-white (single-channel), but were saved as 3-channel RGB files by duplicating the same intensity value into R, G and B.

**Why this hurts training:**
- The model wastes capacity learning from redundant colour channels.
- Data augmentation (HSV jitter, colour noise) adds artefacts that don't exist in real thermal imagery, confusing the model.

**Fix:** Convert every image to true 1-channel grayscale before training.

In [ ]:
# Convert all RGB images in the dataset to true 1-channel grayscale
# Images are overwritten in-place

splits = ["train", "valid", "test"]
total_converted = 0

print("🔄 Starting RGB → Grayscale conversion...")

for split in splits:
    img_dir = Path(DATASET_ROOT) / split / "images"

    if not img_dir.exists():
        print(f"   ⚠️  Skipped '{split}': directory not found at {img_dir}")
        continue

    files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png"))

    if not files:
        print(f"   ⚠️  Skipped '{split}': no images found.")
        continue

    print(f"   Processing '{split}' ({len(files)} images)...")

    for img_path in files:
        try:
            img = Image.open(img_path)
            if img.mode == "RGB":
                img.convert("L").save(img_path)  # L = 8-bit grayscale
                total_converted += 1
        except Exception as e:
            print(f"   ❌ Error on {img_path.name}: {e}")

print(f"\n✅ Done — {total_converted} images converted to true 1-channel grayscale.")

🔄 Starting RGB → Grayscale conversion...
   Processing 'train' (697 images)...
   Processing 'valid' (149 images)...
   Processing 'test' (149 images)...

✅ Done — 995 images converted to true 1-channel grayscale.


In [7]:
# Verify dataset split sizes and confirm images are truly grayscale

print("📂 Dataset split sizes:")
for split in ["train", "valid", "test"]:
    img_dir = Path(DATASET_ROOT) / split / "images"
    images = list(img_dir.rglob("*.jpg")) + list(img_dir.rglob("*.png"))
    print(f"   {split:6s}: {len(images)} images")

# Spot-check one image to confirm channel count
sample_img_path = next((Path(DATASET_ROOT) / "train" / "images").rglob("*.jpg"))
img = Image.open(sample_img_path)
channels = len(img.getbands())
print(f"\n🔍 Sample image: {sample_img_path.name}")
print(f"   Size: {img.width} x {img.height} px")
print(f"   Channels: {channels} ({'✅ Grayscale' if channels == 1 else '❌ Still RGB — rerun conversion'})")

📂 Dataset split sizes:
   train : 697 images
   valid : 149 images
   test  : 149 images

🔍 Sample image: video_013_event7_A007_740_t-00240s_jpg.rf.9a7dd7ac2122dbc08adc0b8ca0b4a8bd.jpg
   Size: 448 x 448 px
   Channels: 1 (✅ Grayscale)


---
## M0 - Binary Detection

### Label transformation

- Class 0 (background) stays 0
- All other classes → 1 (flukeprint)
- Writes to a new dataset folder, never overwrites the original

In [12]:
# ── config ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT  = Path("dataset_M0")
SPLITS       = ["train", "valid", "test"]
# ─────────────────────────────────────────────────────────────────────────────


def transform_label_file(src: Path, dst: Path) -> tuple[int, int]:
    """
    Read a YOLOv8 polygon label file, remap classes, write to dst.
    Returns (total_annotations, remapped_count).
    """
    lines = src.read_text().splitlines()
    out_lines = []
    total = remapped = 0

    for line in lines:
        line = line.strip()
        if not line:
            out_lines.append(line)
            continue

        parts = line.split()
        cls = int(parts[0])
        total += 1

        if cls != 0:
            parts[0] = "1"
            remapped += 1

        out_lines.append(" ".join(parts))

    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text("\n".join(out_lines))
    return total, remapped


def main():

    grand_total = grand_remapped = 0

    for split in SPLITS:
        img_src = DATASET_ROOT / split / "images"
        lbl_src = DATASET_ROOT / split / "labels"
        img_dst = OUTPUT_ROOT  / split / "images"
        lbl_dst = OUTPUT_ROOT  / split / "labels"

        # ── images: copy as-is ───────────────────────────────────────────────
        if img_src.exists():
            if img_dst.exists():
                shutil.rmtree(img_dst)
            shutil.copytree(img_src, img_dst)
            n_imgs = len(list(img_dst.iterdir()))
            print(f"[{split}] copied {n_imgs} images")
        else:
            print(f"[{split}] no images folder found, skipping")

        # ── labels: remap classes ────────────────────────────────────────────
        if not lbl_src.exists():
            print(f"[{split}] no labels folder found, skipping")
            continue

        lbl_dst.mkdir(parents=True, exist_ok=True)
        split_total = split_remapped = 0

        for lbl_file in sorted(lbl_src.glob("*.txt")):
            t, r = transform_label_file(lbl_file, lbl_dst / lbl_file.name)
            split_total    += t
            split_remapped += r

        grand_total    += split_total
        grand_remapped += split_remapped
        print(f"[{split}] {split_total} annotations — "
              f"{split_remapped} remapped to class 1, "
              f"{split_total - split_remapped} kept as class 0")

    # ── copy + patch yaml if present ─────────────────────────────────────────
    for yaml_file in DATASET_ROOT.glob("*.yaml"):
        content = yaml_file.read_text()
        content = re.sub(r"nc\s*:\s*\d+", "nc: 2", content)
        content = re.sub(
            r"names\s*:.*?(?=\n\S|\Z)",
            "names:\n  0: background\n  1: flukeprint",
            content,
            flags=re.DOTALL,
        )
        dst_yaml = OUTPUT_ROOT / yaml_file.name
        dst_yaml.parent.mkdir(parents=True, exist_ok=True)
        dst_yaml.write_text(content)
        print(f"[yaml] patched and copied → {dst_yaml}")

    print(f"\n✓ Done. Output: {OUTPUT_ROOT.resolve()}")
    print(f"  Total annotations : {grand_total}")
    print(f"  Remapped → class 1: {grand_remapped}")
    print(f"  Kept as class 0   : {grand_total - grand_remapped}")


if __name__ == "__main__":
    main()

[train] copied 697 images
[train] 2413 annotations — 2171 remapped to class 1, 242 kept as class 0
[valid] copied 149 images
[valid] 514 annotations — 444 remapped to class 1, 70 kept as class 0
[test] copied 149 images
[test] 513 annotations — 453 remapped to class 1, 60 kept as class 0
[yaml] patched and copied → dataset_M0/data.yaml

✓ Done. Output: /home/floreM/FlukePrint_YOLO/behav_model/dataset_M0
  Total annotations : 3440
  Remapped → class 1: 3068
  Kept as class 0   : 372


### M0 Training

In [17]:
# ── Training with best hyperparameters ──────────────────────────────────
# Load the best hyperparameters found by the tuner and train.

# ── config ────────────────────────────────────────────────────────────────────
M0_DATASET  = Path("dataset_M0")
M0_YAML     = M0_DATASET / "data.yaml"
# ─────────────────────────────────────────────────────────────────────────────

with open(MODEL_PARAMS_PATH) as f:
    best_params = yaml.safe_load(f)

model_tuned = YOLO(MODEL_TUNED_PATH)

In [18]:


model_tuned.train(
    data=M0_YAML,
    epochs=150,        
    imgsz=448,
    patience=20,
    name="train_M0",
    **best_params      
)


New https://pypi.org/project/ultralytics/8.4.48 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00784, box=5.14291, cache=False, cfg=None, classes=None, close_mosaic=6, cls=0.55957, compile=False, conf=None, copy_paste=0.00314, copy_paste_mode=flip, cos_lr=False, cutmix=0.00571, data=dataset_M0/data.yaml, degrees=0.00017, deterministic=True, device=None, dfl=1.36906, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.48872, flipud=0.0018, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01493, hsv_s=0.64776, hsv_v=0.27383, imgsz=448, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00218, lrf=0.02127, mask_ratio=4, max_det=300, mixup=0.00285, mode=train, model=./hyper_tuned_

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f9c1dee40d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

### M0 Evaluation

In [ ]:
M0_BEST = Path("runs/segment/train_M0/weights/best.pt")  

final_model = YOLO(M0_BEST)

metrics = final_model.val(
    data=M0_YAML,
    split="val",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True
)

print("\n" + "=" * 60)
print("M0 VALIDATION RESULTS — Tuned Model (best.pt)")
print("=" * 60)
print(f"mAP50        (Box):  {metrics.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics.box.map:.4f}")
print(f"mAP50        (Mask): {metrics.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics.seg.map:.4f}")
print(f"Recall       (Mask): {metrics.seg.r.mean():.4f}")
print(f"Precision    (Mask): {metrics.seg.p.mean():.4f}")
print("=" * 60)


Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,780,374 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 917.1±233.1 MB/s, size: 11.1 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/behav_model/dataset_M0/valid/labels.cache... 149 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 56.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 40% ━━━━╸─────── 4/10 1.3s/it 32.0s<7.5sWARNING ⚠️ NMS time limit 2.800s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 4.5s/it 44.7s.2s
                   all        149        514      0.763      0.737      0.788      0.653      0.745      0.708      0.766      0.558
            background         48      

In [ ]:
# ── Final Test Evaluation — M0 ───────────────────────────────────────────────

M0_BEST = Path("runs/segment/train_M0/weights/best.pt")

final_model = YOLO(M0_BEST)

metrics_test = final_model.val(
    data=M0_YAML,
    split="test",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True
)

p  = metrics_test.seg.p.mean()
r  = metrics_test.seg.r.mean()
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

print("\n" + "=" * 60)
print("M0 TEST RESULTS — Final Held-Out Evaluation")
print("=" * 60)
print(f"mAP50        (Box):  {metrics_test.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics_test.box.map:.4f}")
print(f"mAP50        (Mask): {metrics_test.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics_test.seg.map:.4f}")
print(f"Recall       (Mask): {r:.4f}")
print(f"Precision    (Mask): {p:.4f}")
print(f"F1           (Mask): {f1:.4f}")
print("=" * 60)
print("✓ M0 baseline locked. Safe to proceed to M1.")